### Experimental Setup: Simulated Markov Process from Kernel $P$
We simulate a first-order Markov process in $\mathbb{R}^{d}$, where the transition distribution is a linear Gaussian:

$$
X_{n+1} \mid X_n = x \;\sim\; \mathcal{N}(A x + b, \Sigma)
$$


This defines a **contracting linear Gaussian** transition kernel with known stationary distribution.



### Closed-form Conditional Score

Because the conditional distribution is Gaussian, the conditional log density has a known closed-form gradient:

$$
\nabla_y \log p(y \mid x) = -\Sigma^{-1} (y - A x - b)
$$

This gives us the exact ground-truth **score function** for every transition pair $(x, y)$.


In [1]:
from tqdm.notebook import tqdm
import functools
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pdb

import functools
from torch.optim import Adam
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import tqdm
from tqdm import tqdm
import matplotlib.pyplot as plt
import random

from torch.utils.data import TensorDataset, DataLoader

from torchvision.datasets import FashionMNIST, MNIST
from torchvision import transforms
from torch.utils.data import ConcatDataset
from torch.utils.data import Subset
from scipy import integrate
from torchvision.utils import make_grid

# Set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [2]:
def sample_markov_chain(n_steps, A, b, Sigma, x0=None, seed=None, show_progress=False):
    """
    Generate a sample path from a Gaussian Markov chain:
        X_{t+1} | X_t ~ N(A X_t + b, Sigma)

    Parameters
    ----------
    n_steps : int
        Number of transitions (path will have length n_steps+1).
    A : np.ndarray (d x d)
        Linear transformation matrix.
    b : np.ndarray (d,)
        Bias vector.
    Sigma : np.ndarray (d x d)
        Covariance matrix (positive definite).
    x0 : np.ndarray (d,), optional
        Initial state. Defaults to zero vector.
    seed : int, optional
        Random seed for reproducibility.
    show_progress : bool, optional
        If True, show tqdm progress bar.

    Returns
    -------
    X : np.ndarray of shape (n_steps+1, d)
        The simulated Markov chain sample path.
    """
    rng = np.random.default_rng(seed)
    d = A.shape[0]
    A = np.array(A, dtype=np.float32)
    b = np.array(b, dtype=np.float32)
    Sigma = np.array(Sigma, dtype=np.float32)
    x = np.array(x0, dtype=np.float32) if x0 is not None else np.zeros_like(b, dtype=np.float32)


    # Initialize
    if x0 is None:
        x = np.zeros(d)
    else:
        x = np.array(x0)

    X = [x]
    iterator = range(n_steps)
    if show_progress:
        iterator = tqdm(iterator, desc="Simulating Markov chain")

    for _ in iterator:
        noise = rng.multivariate_normal(mean=np.zeros(d, dtype=np.float32), cov=Sigma)
        x = A @ x + b + noise
        X.append(x)

    return np.array(X)


### Synthetic Markov Process Parameters

To define a stable Markov process in $\mathbb{R}^{d}$, we generate:

- **Transition matrix \(A\)**:  
  We sample 10 random eigenvalues uniformly from $(-1, 1)$, and construct a matrix $A = V D V^{-1}$, where $D$ is the diagonal matrix of eigenvalues. Here $V$ is a random invertible matrix. This all the eigenvalues $|\lambda_i| < 1$ so the process is stable and stationary.

- **Bias vector $b$**:  
  We sample $b \sim \mathcal{U}(-1, 1)^d$ to allow arbitrary mean shift.

- **Noise covariance $\Sigma$**:  
  We generate a random matrix $R$ and define $\Sigma = R R^\top + \delta I$, with small $\delta > 0$, to ensure $\Sigma$ is positive definite.


In [3]:
d =10
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

# Generate matrix A: with all the eigenven value |\lambda_i|<1. A= VDV^{-1}

# Step 1: get random eigen values
eigs = 1.6*torch.rand(d) - 0.8   # uniform(-0.8,0.8), avoid getting eigen values ~ 1, 
D = torch.diag(eigs)

# Step 2: random invertible matrix V
V = torch.randn(d, d)
while torch.linalg.matrix_rank(V) < d:
    V = torch.randn(d, d)
    
# Step 3: construct A with VDV^{-1}
A_P = V @ D @ torch.linalg.inv(V)

# Generate bias b_P
b_P = np.random.rand(d)  # fills b_P with random values between 0 and 1

# Generate covariance matrix Sigma: must be positive definite. Sigma = RR' + \delta I_d 
R = torch.randn(d, d)
delta = 5e-2  # small regularization term, make sure the Sigma is strictly p.d.
Sigma_P = R @ R.T + delta * torch.eye(d)  

# check stability
rho_A = torch.max(torch.abs(torch.linalg.eigvals(A_P))).item()
print("The largest abs eigenvalue of A:", rho_A, "Therefore, the markov chain is stable.")

eigvals_Sigma = torch.linalg.eigvalsh(Sigma_P)
min_eig_Sigma = eigvals_Sigma.min().item()
print("Minimum eigenvalue of Sigma:", min_eig_Sigma, "Therefore, the covariance matrix is p.d.")  # should be > 0

n_pre = 500000

X = sample_markov_chain(
    n_steps=n_pre,
    A=A_P,
    b=b_P,
    Sigma=Sigma_P,
    seed=seed,
    show_progress=True  # <--- tqdm on
)

print("Full path shape:", X.shape)

# Format parameter values into the filename
A_str = f"{rho_A:.2f}"               # the largest eigen value
b_str = f"{np.linalg.norm(b_P):.2f}"  # save 2 digit
Sigma_str = f"{min_eig_Sigma:.2f}"       # smallest eigen value of Sigma

filename = f"messy_markov_path_A_{A_str}_b_{b_str}_Sigma_{Sigma_str}.pt"
torch.save(torch.from_numpy(X), filename)
print("Saved to:", filename)

The largest abs eigenvalue of A: 0.7348888516426086 Therefore, the markov chain is stable.
Minimum eigenvalue of Sigma: 0.05480724573135376 Therefore, the covariance matrix is p.d.


Simulating Markov chain: 100%|██████████████████████████████████████| 500000/500000 [00:33<00:00, 15002.05it/s]


Full path shape: (500001, 10)
Saved to: messy_markov_path_A_0.73_b_1.90_Sigma_0.05.pt


In [4]:
kernel_params = {
    "A": A_P,
    "b": b_P,
    "Sigma": Sigma_P
}
torch.save(kernel_params, f"messy_kernel_params_A_{A_str}_b_{b_str}_Sigma_{Sigma_str}.pt")  
print("Saved kernel parameters.")

Saved kernel parameters.


### Previous Approach 1: **GBRBM (Gaussian–Bernoulli Restricted Boltzmann Machine)**

- **Purpose**: Models the marginal distribution $ p(x) $
- **Assumes**: i.i.d. data $ x \sim p(x) $
- **Score learned**: $ \nabla_x \log p(x) $
- **Limitation**: Cannot model conditional transitions $ p(y \mid x) $, and not suitable for Markov chains

###  Previous Approach 2: **Denoising Score Matching (DSM)**

- **Purpose**: Learns $ \nabla_x \log p(x) $ from noisy versions of $ x $
- **Assumes**: Original data lies on a low-dimensional manifold (e.g., images)
- **Adds noise**: Artificial Gaussian noise $ x \rightarrow x + \sigma \epsilon $
- **Limitation**: My data is from a synthetic Markov process with full support in $ \mathbb{R}^d $, so noise is unnecessary and harmful

---

### Current Goal: **Conditional Score Learning**

- **Goal**: Learn $ \nabla_y \log p(y \mid x) $
- **Data**: $ (x, y) \sim \text{Markov process} $
- **Apparent Training loss**: 
  $$
  J(\theta) = \frac{1}{2} \mathbb{E}_{(x, y)} \left\| \psi_\theta(y, x) - \nabla_y \log p(y \mid x) \right\|^2
  $$
- **Actual Training loss**: 
  $$
      \tilde{J}(\theta) \triangleq \mathbb{E}_{(x, y)} \bigg( \frac{1}{2}\left\|  \psi_\theta(y, x)  \right\|^2 + \nabla_y \cdot \psi_\theta(y,x) \bigg)
  $$
  where $\nabla_y \cdot \psi_\theta(y,x)$ is the divergence of $\psi_\theta(y,x)$ w.r.t $y$, which is exactly the trace of the Jacobian of $\psi_\theta(y,x)$.
- **Advantages**:
  - I know the true conditional score (Gaussian kernel)
  - No need for noise, latent variables, or diffusion
  - Clean supervised setup, tractable and verifiable



### Conditional Score Network: Learning $\nabla_y \log p(y \mid x)$
Build a **feedforward MLP** that maps $(x, y) \in \mathbb{R}^{2d} \to \nabla_y \log p(y \mid x) \in \mathbb{R}^{d}$ and train using the loss above.
We aim to learn the conditional score function of a Markov process, i.e., the gradient of the log transition density:

$$
\psi(y, x) \approx \nabla_y \log p(y \mid x)
$$

#### Transition from Unconditional to Conditional

| Setting            | Unconditional Score Matching        | Conditional Score Matching                   |
|-------------------|-------------------------------------|----------------------------------------------|
| Input to network  | $x \in \mathbb{R}^d$              | $(y, x) \in \mathbb{R}^{2d}$                |
| Output of network | $\nabla_x \log p(x) \in \mathbb{R}^d$ | $\nabla_y \log p(y \mid x) \in \mathbb{R}^d$ |


#### Training Objective

We train the network using **score matching**, the assumption that:

- The Markov chain is **stationary** after burn-in,
- Each sample pair $(X_{n-1}, X_n)$ is drawn from the stationary joint distribution.

This framework enables us to detect changes in the Markov transition kernel by comparing learned score functions before and after the change.




### Learning and Evaluation

We train a neural network to learn the score function:

$$
\psi(y, x) \approx \nabla_y \log p(y \mid x) =s_{\text{true}}(y,x)
$$

To assess learning quality, we compute the **mean squared error (MSE)** between the trained network output and the closed-form score:

$$
\text{MSE} = \frac{1}{N} \sum_{n=1}^N \left\| \psi(X_n, X_{n-1}) + \Sigma^{-1}(X_n - A X_{n-1} - b) \right\|^2
$$

$$
  \text{VarScale} = \frac{1}{N}\sum_{n=1}^N |s_{\text{true}}(X_n,X_{n-1})|^2.
$$

If the MSE is small, i.e., relative error 
$$  
\frac{\text{MSE} }{\text{VarScale} } < 10^{-3}
$$, 
this indicates that the network has effectively learned the true conditional score.


In [4]:
#@title Define the Network
class ConditionalScoreNet(nn.Module):
    """
    Neural network to approximate conditional score:
        ψ(y, x; θ) ≈ ∇_y log p(y | x)

    Input:  concatenated vector (x, y) ∈ R^{2d}
    Output: vector in R^d
    """

    def __init__(self, d, hidden_dim=512, num_layers=6):
        super().__init__()
        layers = []

        # First layer: input = 2d, hidden_dim
        layers.append(nn.Linear(2*d, hidden_dim))
        layers.append(nn.SiLU())  # smooth activation

        # Middle layers
        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.SiLU())

        # Final layer: hidden_dim → d (score vector dimension)
        layers.append(nn.Linear(hidden_dim, d))

        # Register as Sequential
        self.net = nn.Sequential(*layers)

    def forward(self, x, y):
        """
        Forward pass.
        x: tensor of shape (batch, d)
        y: tensor of shape (batch, d)
        Returns: ψ(y, x; θ) ∈ R^{batch × d}
        """
        inp = torch.cat([x, y], dim=1)  # concatenate along features
        return self.net(inp)


In [5]:
#@title Define the loss function

def hyvarinen_loss(model, x, y):
    """
    Hyvarinen loss for conditional score learning.

    Args:
        model: ConditionalScoreNet
        x: tensor (batch, d)
        y: tensor (batch, d)

    Returns:
        scalar loss
    """
    x.requires_grad_(False)
    y.requires_grad_(True)
    psi = model(x, y)

    loss1 = 0.5 * (psi ** 2).sum(dim=1).mean()

    # Compute divergence ∇_y · ψ
    grads = []
    for i in range(psi.shape[1]):
        grad = torch.autograd.grad(psi[:, i].sum(), y, create_graph=True)[0][:, i]
        grads.append(grad)
    divergence = torch.stack(grads, dim=1).sum(dim=1)

    loss2 = divergence.mean()
    return loss1 + loss2


In [6]:
# Load path
# X = torch.load("markov_path_A0.99_b2.04_Sigma0.60.pt").float()  # shape (T, d)


# Build dataset of pairs, starting from the 50th step
x_data = X[9999:-1]   # X_{n-1}, start at index 9999
y_data = X[10000:]     # X_n, start at index 10000


print("x_data shape:", x_data.shape)  # (T-1, d)
print("y_data shape:", y_data.shape)  # (T-1, d)


x_data shape: (490001, 10)
y_data shape: (490001, 10)


In [ ]:
#@title Train dataset
dataset = TensorDataset(torch.tensor(x_data, dtype=torch.float32).to(device),
                        torch.tensor(y_data, dtype=torch.float32).to(device))


dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

# Parameters
d = X.shape[1]   # dimension of your Markov process
model = ConditionalScoreNet(d=d, hidden_dim=512, num_layers=6).to(device)  # using GPU
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)

# Training loop
num_epochs = 50
# Outer loop with tqdm progress bar
for epoch in tqdm(range(num_epochs)):
    total_loss = 0
    for x_batch, y_batch in dataloader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
    
        optimizer.zero_grad()
        loss = hyvarinen_loss(model, x_batch, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    tqdm.write(f"[Epoch {epoch+1}] Average Loss: {avg_loss:.6f}")


  0%|                                                                                           | 0/50 [00:00<?, ?it/s]

In [13]:
# Save the trained model parameters in src directory
model_path = f"messy_markov_model_A_{A_str}_b_{b_str}_Sigma_{Sigma_str}.pth"
torch.save(model.state_dict(), model_path)
print(f"Model saved to {model_path}")

Model saved to markov_model_A_0.94_b_1.68_Sigma_0.01.pth


### Stationary Distribution

This Markov chain is geometrically ergodic and converges to its stationary distribution:

- Mean: $ \mu_\infty = (I - A_0)^{-1} b_0 = \mathbf{0} $
- Covariance:$ \Sigma_\infty = A \Sigma_\infty A^\top + \Sigma $.  Here, we have  
$
\Sigma_\infty = a^2 \Sigma_\infty + \Sigma
\Rightarrow \Sigma_\infty (1 - a^2) = \Sigma
\Rightarrow \Sigma_\infty = \frac{\Sigma}{1 - a^2}
$


### Mixing Time

The covariance error decays as:

$$
\| \Sigma_n - \Sigma_\infty \| \le \rho(A_0)^{2n} \cdot \| \Sigma_0 - \Sigma_\infty \|
$$

For a tolerance of $ \varepsilon = 10^{-4} $, we solve:

$$
0.99^{2n}  \le 10^{-4} \quad \Rightarrow \quad n \ge 459
$$

Thus, we assume after **459 samples**, the data are drawn from the stationary distribution.

## Why Conditional Score Learning ≠ Learning $\nabla_x \log \pi(x)$

**Concern:**  
Since the Markov chain is stationary, all $X_n \sim \pi_P$.  
Does this mean the network only learns $\nabla_x \log \pi_P(x)$?

**Answer:**  
No — because the training objective uses *pairs* $(X_{n-1}, X_n)$, not singletons $X_n$.  

- The conditional score is defined by  
  $$
  \nabla_y \log p(y \mid x) = \nabla_y \log p(x,y),
  $$
  i.e. derivative of the **joint density** w.r.t. the future state $y$.
- Training on pairs with score matching aligns $\psi(x,y)$ with this conditional gradient, not the stationary score $\nabla_x \log \pi(x)$.

**Numerical evidence:**  
- I trained on stationary pairs $(x,y)$, with $n\ge 10000$.  
- I then tested on the **first 459 pre-mixing pairs** (non-stationary region).  
- Result: MSE $\approx 10^{-3}$, showing the model generalized to early transitions.  

👉 This confirms the network truly learned $\nabla_y \log p(y \mid x)$ (the transition dynamics), not merely $\nabla_x \log \pi(x)$ (the stationary marginal).



In [13]:

# Select test set
X_test_x = torch.tensor(X[:999], dtype=torch.float32).to(device)   # X_{n-1}
X_test_y = torch.tensor(X[1:1000], dtype=torch.float32).to(device) # X_n

N = X_test_x.shape[0]

# Convert kernel parameters to tensors
A = torch.tensor(A_P, dtype=torch.float32, device=device)
b = torch.tensor(b_P, dtype=torch.float32, device=device)
Sigma = torch.tensor(Sigma_P, dtype=torch.float32, device=device)
Sigma_inv = torch.linalg.inv(Sigma)

# Compute true score
with torch.no_grad():
    mu = X_test_x @ A.T + b  # shape (N, d)
    true_score = - (X_test_y - mu) @ Sigma_inv.T  # ∇_y log p(y|x)

# Compute predicted score
model.eval()
with torch.no_grad():
    pred_score = model(X_test_x, X_test_y)  # shape (N, d)

# === Metrics ===
# MSE = || pred - true ||^2
mse = torch.mean((pred_score - true_score)**2).item()

# VarScale = || true ||^2
varscale = torch.mean(true_score**2).item()

# Relative error
relative_error = mse / varscale

# === Report ===
print(f"MSE        = {mse:.6e}")
print(f"VarScale   = {varscale:.6e}")
print(f"Rel. Error = {relative_error:.6e}")


MSE        = 1.293524e-01
VarScale   = 1.012864e+01
Rel. Error = 1.277095e-02
